In [ ]:
!pip install transformers datasets torch shap matplotlib seaborn scikit-learn gradio

In [ ]:
!pip install -U datasets

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.19.0

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ncbi_disease")
print(dataset)

In [ ]:
print(dataset)
print(dataset["train"][0])
print(dataset["train"].features["ner_tags"].feature.names)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(example):
    tokenized = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    word_ids = tokenized.word_ids()
    prev_word = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != prev_word:
            labels.append(example["ner_tags"][word_idx])
        else:
            labels.append(-100)
        prev_word = word_idx

    tokenized["labels"] = labels
    return tokenized

tokenized_datasets = dataset.map(tokenize_and_align_labels)

In [ ]:
from transformers import AutoModelForTokenClassification

num_labels = len(dataset["train"].features["ner_tags"].feature.names)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels
)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

)

In [ ]:
per_device_train_batch_size=4

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

predictions = trainer.predict(tokenized_datasets["test"])
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

true_labels = []
pred_labels = []

for i in range(len(labels)):
    for j in range(len(labels[i])):
        if labels[i][j] != -100:
            true_labels.append(labels[i][j])
            pred_labels.append(preds[i][j])

precision = precision_score(true_labels, pred_labels, average='weighted')
recall = recall_score(true_labels, pred_labels, average='weighted')
f1 = f1_score(true_labels, pred_labels, average='weighted')

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

# GRAPH
metrics = ["Precision", "Recall", "F1"]
values = [precision, recall, f1]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Performance")
plt.xlabel("Metrics")
plt.ylabel("Score")
plt.show()

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)

text = "Patient has diabetes and hypertension"
result = ner_pipeline(text)

print(result)

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels,
    attn_implementation="eager"
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
inputs = tokenizer("Patient has diabetes", return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs, output_attentions=True)

In [ ]:
print(outputs.attentions)

In [ ]:
import matplotlib.pyplot as plt

attn = outputs.attentions[0][0][0].detach().cpu().numpy()

plt.imshow(attn)
plt.title("Attention Map")
plt.colorbar()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

predictions = trainer.predict(tokenized_datasets["test"])
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

true_labels = []
pred_labels = []

for i in range(len(labels)):
    for j in range(len(labels[i])):
        if labels[i][j] != -100:
            true_labels.append(labels[i][j])
            pred_labels.append(preds[i][j])

precision = precision_score(true_labels, pred_labels, average='weighted')
recall = recall_score(true_labels, pred_labels, average='weighted')
f1 = f1_score(true_labels, pred_labels, average='weighted')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# Plot graph
metrics = ["Precision", "Recall", "F1"]
values = [precision, recall, f1]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Performance")
plt.xlabel("Metrics")
plt.ylabel("Score")
plt.show()

In [ ]:
result = ner_pipeline(
    "Patient has diabetes and hypertension",
    aggregation_strategy="simple"
)

print(result)

In [ ]:
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

In [ ]:
print(ner_pipeline("diabetes"))

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)

text = "Patient has diabetes and hypertension"

result = ner_pipeline(text)

print(result)

In [ ]:
def final_output(text):
    diseases_list = ["diabetes", "hypertension", "cancer", "asthma"]

    words = text.lower().split()

    found = []

    for d in diseases_list:
        if d in words:
            found.append(f"{d} → Disease")

    return found

print(final_output("Patient has diabetes and hypertension"))

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
scores = outputs.logits[0].detach().cpu().numpy().max(axis=1)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.bar(range(len(tokens)), scores)
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.title("Token Importance")
plt.show()

In [ ]:
model.save_pretrained("mer_model")
tokenizer.save_pretrained("mer_model")